# 21 — Paper 3: analyses required by the thesis proposal (section 7.3)

Adds the parts of proposal section 7.3 that the Paper 3 manuscript (20261104) does not yet contain. Uses the **same input files as Notebook 20** and the same primary period (January 2022 – November 2024, December excluded, satellite PM2.5 only).

| Proposal item | Section | Output file (`OUTDIR`) |
|---|---|---|
| 7.3.3 rates at 5 time scales + seasonal Mann–Kendall | 1 | `T_rates_timescales.csv`, `T_trend_mk.csv` |
| 7.3.2 (2) PM2.5 hotspot (Gi*, LISA) by fiscal year and season | 2 | `T_pm25_hotspots.csv` |
| 7.2.4 / 7.3.2 (3) source-sector position from Paper 2 CPF, distance to 9 cement plants, DIW counts in 3/5/10/15 km, distance-decay λ, highway distance | 3 | `T_tambon_exposure_indicators.csv` |
| 7.3.5 step 1 Spearman PM2.5 / source distance vs rate and SIR | 4 | `T_spearman.csv` |
| 7.3.5 step 2 bivariate Moran's I and bivariate LISA | 5 | `T_bivariate_moran.csv`, `T_bivariate_lisa_HH.csv` |
| 7.3.5 step 3 rate ratios by exposure zone | 6 | `T_rate_ratios.csv` |
| 7.3.5 step 4 per-disease within-tambon IRR, PM2.5 × season, effect modification by source indicators | 7 | `T_irr_by_disease.csv`, `T_irr_season.csv`, `T_effect_modification.csv` |
| 7.3.5 step 5 MAE (helper for the Notebook 20 prediction output) | 8 | — |
| Summary of all numbers to paste into the manuscript | 9 | `SUMMARY_for_manuscript.txt` |

**Source sector (Paper 2, current version):** elevated CPF at both stations for wind sectors centred 78.75°–146.25° (edges 67.5°–157.5°); CPF maximum 123.75° at 24T and 101.25° at 25T. Following proposal 7.2.4, a tambon is *in the source sector* when the bearing from its nearest station (24T or 25T) to its interior point lies in 67.5°–157.5°. The facility-downwind indicator of the manuscript is recomputed with the reversed sector (247.5°–337.5° from the facility, within 5 km).

Step-4 models here are negative binomial with tambon fixed effects (the same within-tambon estimator as the manuscript's sensitivity analyses). `Runtime → Run all` takes about 5–15 min.

> **แก้ไข 8 พ.ย. 2569:** การควบคุมเวลาทุกจุดเปลี่ยนจากปีปฏิทินเป็น **ปีงบประมาณ** (`D["fy"]`, ต.ค.–ก.ย.): rate รายปี, hotspot รายปี, Spearman รายปี, rate ratio (strata) และ NB fixed effect (`C(fy)`) — ผลเดิมถูกล้างแล้ว ต้อง `Runtime → Run all` ใหม่ · ช่วงหลัก ม.ค. 2565 – พ.ย. 2567 มีปีงบ 2565 (9 เดือน), 2566 และ 2567 (11 เดือน) และ 2568 (2 เดือน)

## 0 · Config

In [ ]:
# ===== Repository data (data/processed/) =====
# Opened from GitHub in Colab: the repository is cloned so that data/processed/ is available.
# Run locally: start Jupyter in the repository (or its notebooks/ folder).
# If data/processed/ is not found, the notebook falls back to the original Google Drive paths.
import os, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-acute-illness-saraburi"
USE_REPO = True          # False = use the original Google Drive paths only
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-acute-illness-saraburi"):
    if os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if USE_REPO and REPO_DIR is None and os.path.isdir("/content"):
    REPO_DIR = "/content/paper3-pm25-acute-illness-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
REPO_DATA = os.path.join(REPO_DIR, "data", "processed") if REPO_DIR else ""
USE_REPO_DATA = bool(USE_REPO and REPO_DATA and os.path.isdir(REPO_DATA))

def repo_file(name):
    """Path of a file in data/processed/ (matched by file name), or None."""
    p = os.path.join(REPO_DATA, os.path.basename(str(name))) if USE_REPO_DATA else ""
    return p if p and os.path.exists(p) else None

print("Repository data:", REPO_DATA if USE_REPO_DATA else "not found - using Google Drive paths")

In [ ]:
DATA_DIR      = "/content/drive/MyDrive/PM25_Saraburi_notebooks"     # same as Notebook 20
PM_PANEL_CSV  = "pm25_tambon_monthly_fy2565_2569.csv"
BOUNDARY_GPKG = "sat_pm25_tambon_summary.gpkg"
HDC_MONTHLY_CSV = "000_research/11HealthImpactAssessment_SW/0data/hdc_saraburi_pm25_disease_tambon_monthly_2565_2569.csv"
HDC_YEAR_IS_FISCAL = False
DIW_CSV  = "3-ColabProjects/Paper2/02_Data/processed/diw_facilities_geocoded.csv"   # 295 facilities (Paper 2 / Paper 3)
ROADS_CANDIDATES = ["3-ColabProjects/Paper2/02_Data/raw/Paper1_OSM_Highway1_2.geojson",     # Highway 1 and 2 (same file as Papers 1-2)
                    "3-ColabProjects/Paper1/02_Data/processed/Paper1_OSM_Highway1_2.geojson",
                    "3-ColabProjects/Paper2/02_Data/raw/osm_highway_1_2.geojson"]
ROADS_GEOJSON = ROADS_CANDIDATES[0]                                      # OSM download is only a last resort
START, END = "2022-01", "2024-11"          # primary period of the manuscript
DROP_DECEMBER = True
PM_UNIT = 10
SECTOR = (67.5, 157.5)                     # elevated-CPF sector edges, Paper 2 (Table 4)
STATIONS = {"24T": (14.685833, 100.871996), "25T": (14.526343, 100.926047)}
PLANTS = {"TPI Polene Mittraphap": (14.646310, 101.126496), "Siam Cement Kaeng Khoi": (14.647975, 101.039987),
          "Siam Cement Khao Wong": (14.674128, 100.851875), "Siam Cement Ta Luang": (14.570239, 100.763862),
          "Siam City Thap Kwang 1": (14.627492, 101.087613), "Siam City Thap Kwang 2": (14.632372, 101.077219),
          "Siam City Thap Kwang 3": (14.631969, 101.098706), "Thai Pride Hin Son": (14.737328, 101.038138),
          "Asia Cement Phu Khang": (14.673757, 100.803369)}         # Global Energy Monitor (2026), as in Papers 1-2
BUFFERS_KM = [3, 5, 10, 15]; LAMBDAS_KM = [1, 2, 5]
PERM = 999; SEED = 2026
SAVE_TO_DRIVE = True

In [ ]:
# ===== Use data/processed/ from the repository when available (the Drive paths above are the fallback) =====
if USE_REPO_DATA:
    DATA_DIR = REPO_DATA
    for _v in ("PM_PANEL_CSV", "SAT_MONTHLY_CSV", "BOUNDARY_GPKG", "HDC_MONTHLY_CSV", "DIW_CSV", "ROADSIDE_CSV"):
        if globals().get(_v): globals()[_v] = os.path.basename(globals()[_v])
    if "ROADS_CANDIDATES" in globals():
        ROADS_GEOJSON = "osm_highway_1_2.geojson"; ROADS_CANDIDATES = [ROADS_GEOJSON]
    if "ERA5_CACHE_DIR" in globals():
        ERA5_CACHE_DIR = os.path.join(REPO_DATA, "ERA5_cache")   # hourly ERA5 2022–2024 (Open-Meteo archive); other years are downloaded
    SAVE_TO_DRIVE = False                                  # outputs go to the working folder, not data/processed/
    print("Using repository data in", DATA_DIR)

In [ ]:
import importlib.util, subprocess, sys
for m, p in {"libpysal": "libpysal", "esda": "esda", "pymannkendall": "pymannkendall", "pyproj": "pyproj"}.items():
    if importlib.util.find_spec(m) is None: subprocess.run(f"{sys.executable} -m pip -q install {p}", shell=True)
import os, json, re, warnings, numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import statsmodels.api as sm, statsmodels.formula.api as smf
from scipy import stats
import libpysal, esda, pymannkendall as mk
from libpysal.weights import Queen
from pyproj import Geod
warnings.filterwarnings("ignore"); pd.set_option("display.width", 170)
geod = Geod(ellps="WGS84"); UTM = "EPSG:32647"
IN_COLAB = os.path.isdir("/content")
if IN_COLAB and DATA_DIR.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
DATA_DIR = os.environ.get("PAPER3_DATA_DIR", DATA_DIR)
for k in ("HDC_MONTHLY_CSV", "DIW_CSV", "ROADS_GEOJSON"):
    if os.environ.get("PAPER3_" + k): globals()[k] = os.environ["PAPER3_" + k]
OUTDIR = os.path.join(DATA_DIR, "paper3", "proposal_alignment") if SAVE_TO_DRIVE else "./proposal_alignment"
os.makedirs(OUTDIR, exist_ok=True)
def _path(p):
    if os.path.isabs(p): return p
    for b in (DATA_DIR, "/content/drive/MyDrive", "."):
        if os.path.exists(os.path.join(b, p)): return os.path.join(b, p)
    return os.path.join(DATA_DIR, p)
def save(df, name):
    df.to_csv(os.path.join(OUTDIR, name), index=False, encoding="utf-8-sig"); return df
SUMMARY = []
def note(s): SUMMARY.append(s); print(s)
print("OUTDIR =", OUTDIR)

## 0b · Data (same files and rules as Notebook 20)

In [ ]:
def _code6(s): return s.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True).str[:6]
COLMAP = {"r1_1": "COPD", "r1_2": "Asthma", "r1_3": "Pneumonia", "r1_4": "Influenza", "r1_5": "Acute pharyngitis",
          "r1_6": "Chronic rhinitis", "r1_7": "Bronchitis", "r2_1": "Ischaemic heart disease", "r2_2": "Stroke",
          "r3": "Eye inflammation", "r4": "Dermatitis"}
DISEASES = list(COLMAP.values())
GROUPS = {"Respiratory": DISEASES[:7], "Cardiovascular": DISEASES[7:9], "Eye": [DISEASES[9]], "Skin": [DISEASES[10]]}
ALIAS = {"Ischemic heart diseases": "Ischaemic heart disease", "Cerebrovascular diseases (Stroke)": "Stroke",
         "กลุ่มโรคตาอักเสบ": "Eye inflammation", "กลุ่มโรคผิวหนังอักเสบ": "Dermatitis"}

PM = pd.read_csv(_path(PM_PANEL_CSV), encoding="utf-8-sig", dtype={"a_code": str}); PM["a_code"] = _code6(PM.a_code)
BND = gpd.read_file(_path(BOUNDARY_GPKG)); BND["a_code"] = _code6(BND.a_code)
BND = BND[["a_code", "name", "pop", "geometry"]].drop_duplicates("a_code").reset_index(drop=True)

H = pd.read_csv(_path(HDC_MONTHLY_CSV), encoding="utf-8-sig", dtype=str)
H.columns = [c.strip() for c in H.columns]
ccode = next(c for c in H.columns if c.lower() in ("a_code", "areacode", "tambon_code", "subdistrict_code", "code"))
H["a_code"] = _code6(H[ccode])
if "ym" in H.columns:
    ym = H.ym.str.replace(r"\D", "", regex=True); y, m = ym.str[:4].astype(int), ym.str[4:6].astype(int)
else:
    y = H[next(c for c in H.columns if c.lower() in ("year", "yr", "b_year"))].astype(float).astype(int)
    m = H[next(c for c in H.columns if c.lower() in ("month", "mon", "b_month"))].astype(float).astype(int)
y = np.where(y > 2400, y - 543, y)
if HDC_YEAR_IS_FISCAL: y = np.where(m >= 10, y - 1, y)
H["ym"] = [f"{a:04d}-{b:02d}" for a, b in zip(y, m)]
H = H.rename(columns={**COLMAP, **ALIAS})
miss = [d for d in DISEASES if d not in H.columns]
if miss: raise ValueError(f"disease columns not found: {miss} — columns are {list(H.columns)[:30]}")
for d in DISEASES: H[d] = pd.to_numeric(H[d], errors="coerce").fillna(0)
H = H.groupby(["a_code", "ym"], as_index=False)[DISEASES].sum()

D = PM[["a_code", "ym", "pm25_sat_pop"]].rename(columns={"pm25_sat_pop": "pm"}).merge(H, on=["a_code", "ym"]).merge(
    BND[["a_code", "name", "pop"]], on="a_code")
D = D[(D.ym >= START) & (D.ym <= END) & D.pm.notna() & (D["pop"] > 0)].copy()
if DROP_DECEMBER: D = D[~D.ym.str.endswith("-12")]
t = pd.to_datetime(D.ym); D["year"] = t.dt.year; D["month"] = t.dt.month
D["fy"] = D.year + 543 + (D.month >= 10).astype(int)   # ปีงบประมาณ พ.ศ. (ต.ค.–ก.ย.)
D["season"] = D.month.map(lambda k: "Winter" if k in (11, 12, 1, 2) else ("Summer" if k in (3, 4, 5) else "Rainy"))
D["period2"] = np.where(D.fy <= 2566, "FY2565–2566", "FY2567–2568")
for g, cols in GROUPS.items(): D[g] = D[cols].sum(axis=1)
D["pm_w"] = (D.pm - D.groupby("a_code").pm.transform("mean")) / PM_UNIT
D["log_pop"] = np.log(D["pop"])
OUTCOMES = list(GROUPS) + DISEASES
TAM = BND[BND.a_code.isin(D.a_code.unique())].reset_index(drop=True)
note(f"Data: {D.a_code.nunique()} tambons × {D.ym.nunique()} months ({D.ym.min()}–{D.ym.max()}), visits = {int(D[list(GROUPS)].sum().sum()):,}")

## 1 · Rates at the five time scales of the proposal and seasonal Mann–Kendall (7.3.3)

In [ ]:
MONTHS_PER = {"Overall": D.ym.nunique()}
rows = []
def rate(df, g):          # per 1,000 population per year
    n_m = df.ym.nunique(); pop = df.drop_duplicates("a_code")["pop"].sum()
    return 1000 * df[g].sum() / pop / (n_m / 12)
for g in OUTCOMES:
    rows.append(dict(outcome=g, scale="Overall", level="Jan 2022–Nov 2024", rate=rate(D, g)))
    for col, sc in [("fy", "Fiscal year"), ("period2", "Two-year"), ("season", "Season"), ("month", "Month")]:
        for lv, sub in D.groupby(col): rows.append(dict(outcome=g, scale=sc, level=str(lv), rate=rate(sub, g)))
RT = save(pd.DataFrame(rows), "T_rates_timescales.csv")
display(RT[RT.outcome.isin(GROUPS)].pivot_table(index=["scale", "level"], columns="outcome", values="rate").round(1))
mkrows = []
for g in OUTCOMES:
    s = D.groupby("ym").apply(lambda x: 1000 * x[g].sum() / x["pop"].sum() * 12).sort_index()
    r = mk.seasonal_test(s.values, period=11 if DROP_DECEMBER else 12)
    mkrows.append(dict(outcome=g, trend=r.trend, tau=r.Tau, p=r.p, sen_slope_per_month=r.slope))
MK = save(pd.DataFrame(mkrows), "T_trend_mk.csv"); display(MK.round(3))
for g in GROUPS:
    r = MK.set_index("outcome").loc[g]; note(f"Seasonal Mann–Kendall {g}: tau = {r.tau:.2f}, p = {r.p:.3f}, Sen = {r.sen_slope_per_month:.2f} per 1,000/yr per month")

## 2 · PM2.5 hotspots of tambons (Gi* and LISA, by fiscal year and season)

In [ ]:
W = Queen.from_dataframe(TAM, use_index=False, silence_warnings=True)
if W.islands:   # connect islands to nearest tambon
    cen = TAM.to_crs(UTM).representative_point()
    nb = dict(W.neighbors)
    for i in W.islands:
        d = cen.distance(cen.iloc[i]); d.iloc[i] = np.inf; j = int(d.values.argmin()); nb[i] = [j]; nb[j] = nb[j] + [i]
    W = libpysal.weights.W(nb, silence_warnings=True)
W.transform = "r"
def bh(p):
    p = np.asarray(p); n = len(p); o = np.argsort(p); q = p[o] * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]; out = np.empty(n); out[o] = np.minimum(q, 1); return out
hot = []
for col, lv_list in [("all", ["Jan 2022–Nov 2024"]), ("fy", sorted(D.fy.unique())), ("season", ["Winter", "Summer", "Rainy"])]:
    for lv in lv_list:
        sub = D if col == "all" else D[D[col] == lv]
        x = sub.groupby("a_code").pm.mean().reindex(TAM.a_code).values
        G = esda.G_Local(x, W, star=True, permutations=PERM, seed=SEED, transform="B")
        L = esda.Moran_Local(x, W, permutations=PERM, seed=SEED)
        q_g, q_l = bh(G.p_sim), bh(L.p_sim)
        for i, a in enumerate(TAM.a_code):
            hot.append(dict(period=str(lv), a_code=a, name=TAM.name[i], pm=x[i], gi_z=G.Zs[i], gi_p=G.p_sim[i], gi_q=q_g[i],
                            gi_hot=bool(G.Zs[i] > 1.96 and G.p_sim[i] < 0.05), gi_hot_fdr=bool(G.Zs[i] > 0 and q_g[i] < 0.05),
                            lisa_q=L.q[i], lisa_p=L.p_sim[i], lisa_HH_fdr=bool(L.q[i] == 1 and q_l[i] < 0.05)))
HOT = save(pd.DataFrame(hot), "T_pm25_hotspots.csv")
summ = HOT.groupby("period").agg(n_gi_hot=("gi_hot", "sum"), n_gi_hot_fdr=("gi_hot_fdr", "sum"), n_lisa_HH_fdr=("lisa_HH_fdr", "sum"))
display(summ)
HOTSET = set(HOT[(HOT.period == "Jan 2022–Nov 2024") & HOT.gi_hot].a_code)
pers = HOT[HOT.period.isin([str(y) for y in sorted(D.fy.unique())])].groupby("a_code").gi_hot.sum()
note(f"PM2.5 Gi* hotspots (z > 1.96, p < 0.05), whole period: {len(HOTSET)} tambons; with FDR: {int(summ.loc['Jan 2022–Nov 2024','n_gi_hot_fdr'])}; "
     f"hot in every fiscal year: {int((pers == D.fy.nunique()).sum())}; by season: " + ", ".join(f"{s} {int(summ.loc[s,'n_gi_hot'])}" for s in ["Winter", "Summer", "Rainy"]))
note("Hotspot tambons (whole period): " + ", ".join(sorted(TAM.set_index('a_code').loc[list(HOTSET), 'name'])))

## 3 · Source-position, buffer, distance-decay and highway indicators

In [ ]:
pt = TAM.representative_point(); TLAT, TLON = pt.y.values, pt.x.values
def bearing_dist(lat0, lon0, lat1, lon1):
    az, _, d = geod.inv(np.full(len(lat1), lon0), np.full(len(lat1), lat0), lon1, lat1); return az % 360, d / 1000
IND = pd.DataFrame({"a_code": TAM.a_code, "name": TAM.name})
# (a) proposal 7.2.4: in the elevated-CPF sector of the nearest station
dst = {s: bearing_dist(la, lo, TLAT, TLON) for s, (la, lo) in STATIONS.items()}
near = np.where(dst["24T"][1] <= dst["25T"][1], "24T", "25T")
brg = np.where(near == "24T", dst["24T"][0], dst["25T"][0])
IND["nearest_station"] = near; IND["bearing_from_station"] = brg
IND["source_sector"] = ((brg >= SECTOR[0]) & (brg < SECTOR[1])).astype(int)
# (b) distance to nearest of the 9 integrated cement plants
pd_ = np.c_[[bearing_dist(la, lo, TLAT, TLON)[1] for la, lo in PLANTS.values()]].T
IND["dist_plant_km"] = pd_.min(1)
# (c) DIW facilities: counts in buffers, decay index, updated facility-downwind
DIW = pd.read_csv(_path(DIW_CSV)).dropna(subset=["lat", "lon"])
Fd = np.zeros((len(TAM), len(DIW))); Fb = np.zeros_like(Fd)
for j, (la, lo) in enumerate(zip(DIW.lat, DIW.lon)):
    b, d = bearing_dist(la, lo, TLAT, TLON); Fd[:, j] = d; Fb[:, j] = b
for r in BUFFERS_KM: IND[f"n_fac_{r}km"] = (Fd <= r).sum(1)
for lam in LAMBDAS_KM: IND[f"decay_l{lam}"] = np.exp(-Fd / lam).sum(1)
rev = ((SECTOR[0] + 180) % 360, (SECTOR[1] + 180) % 360)
IND["downwind_fac"] = ((Fd <= 5) & (Fb >= rev[0]) & (Fb < rev[1])).any(1).astype(int)
IND["dist_fac_km"] = Fd.min(1)
# (d) distance to Highway 1 or 2
ROADS_GEOJSON = next((p for p in ([ROADS_GEOJSON] + ROADS_CANDIDATES) if os.path.exists(_path(p))), ROADS_GEOJSON)
if not os.path.exists(_path(ROADS_GEOJSON)):
    import requests, time
    q = '[out:json][timeout:180];(way["highway"~"^(motorway|trunk|primary)$"]["ref"~"^(1|2)$"](14.0,100.4,15.3,101.6););out geom;'
    js = None
    for url in ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter",
                "https://maps.mail.ru/osm/tools/overpass/api/interpreter"]:
        for attempt in range(3):
            try:
                r = requests.post(url, data={"data": q}, timeout=200, headers={"User-Agent": "paper3-notebook"})
                if r.ok and r.text.lstrip().startswith("{"): js = r.json(); break
                print(f"{url}: HTTP {r.status_code}, retry"); time.sleep(10 * (attempt + 1))
            except Exception as e:
                print(f"{url}: {type(e).__name__}, retry"); time.sleep(10 * (attempt + 1))
        if js: break
    if js is None:
        raise RuntimeError("Highway file not found and OpenStreetMap could not be reached. Copy Paper1_OSM_Highway1_2.geojson "
                           "(Paper1/02_Data/processed) into Paper2/02_Data/raw and run this cell again.")
    feats = {}
    for el in js["elements"]: feats.setdefault(el["tags"]["ref"], []).append([[p["lon"], p["lat"]] for p in el["geometry"]])
    os.makedirs(os.path.dirname(_path(ROADS_GEOJSON)), exist_ok=True)
    json.dump({"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"ref": r}, "geometry": {"type": "MultiLineString", "coordinates": c}} for r, c in feats.items()]}, open(_path(ROADS_GEOJSON), "w"))
print("highways:", _path(ROADS_GEOJSON))
RD = gpd.read_file(_path(ROADS_GEOJSON)).set_crs(4326, allow_override=True).to_crs(UTM)
IND["dist_hwy_km"] = gpd.GeoSeries(pt.values, crs=TAM.crs).to_crs(UTM).distance(RD.unary_union).values / 1000
IND["pm25_hotspot"] = IND.a_code.isin(HOTSET).astype(int)
IND = save(IND, "T_tambon_exposure_indicators.csv")
note(f"Source sector (proposal 7.2.4, nearest-station bearing {SECTOR[0]}–{SECTOR[1]}°): {IND.source_sector.sum()} of {len(IND)} tambons; "
     f"updated facility-downwind (≤5 km, {rev[0]}–{rev[1]}° from facility): {IND.downwind_fac.sum()}; "
     f"≥1 DIW facility within 3/5/10/15 km: " + "/".join(str(int((IND[f'n_fac_{r}km'] > 0).sum())) for r in BUFFERS_KM))
display(IND.describe().round(2))

## 4 · Step 1 — Spearman correlation (overall, by fiscal year, by season)

In [ ]:
def tambon_table(sub):
    g = sub.groupby("a_code").agg(pm=("pm", "mean"), pop=("pop", "first"), n=("ym", "nunique"), **{o: (o, "sum") for o in OUTCOMES}).reset_index()
    for o in OUTCOMES:
        g[o + "_rate"] = 1000 * g[o] / g["pop"] / (g.n / 12)
        g[o + "_SIR"] = g[o] / (g["pop"] * g.n * g[o].sum() / (g["pop"] * g.n).sum())
    return g.merge(IND, on="a_code")
SP = []
for col, lvls in [("all", ["Overall"]), ("fy", sorted(D.fy.unique())), ("season", ["Winter", "Summer", "Rainy"])]:
    for lv in lvls:
        T = tambon_table(D if col == "all" else D[D[col] == lv])
        for o in OUTCOMES:
            for xv in ["pm", "dist_plant_km", "dist_fac_km", "dist_hwy_km"]:
                r = stats.spearmanr(T[xv], T[o + "_rate"]); SP.append(dict(period=str(lv), outcome=o, x=xv, rho=r.statistic, p=r.pvalue, n=len(T)))
SP = save(pd.DataFrame(SP), "T_spearman.csv")
display(SP[(SP.period == "Overall") & SP.outcome.isin(GROUPS)].pivot(index="outcome", columns="x", values="rho").round(2))
for o in GROUPS:
    r = SP[(SP.period == "Overall") & (SP.outcome == o) & (SP.x == "pm")].iloc[0]
    note(f"Spearman PM2.5 vs {o} rate (overall): rho = {r.rho:.2f}, p = {r.p:.3f}")

## 5 · Step 2 — bivariate Moran's I and bivariate LISA (PM2.5 × log EB rate)

In [ ]:
def eb_rate(cases, pop):
    r = cases / pop; m = cases.sum() / pop.sum(); v = max(((pop * (r - m) ** 2).sum() / pop.sum()) - m / pop.mean(), 0)
    w = v / (v + m / pop); return w * r + (1 - w) * m
T = tambon_table(D).set_index("a_code").reindex(TAM.a_code).reset_index()
BV, HH = [], []
x = (T.pm - T.pm.mean()) / T.pm.std()
for o in OUTCOMES:
    yv = np.log(eb_rate(T[o].astype(float), (T["pop"] * T.n / 12).astype(float)) * 1000 + 1e-9)
    yv = (yv - yv.mean()) / yv.std()
    mb = esda.Moran_BV(x.values, yv.values, W, permutations=PERM)
    lb = esda.Moran_Local_BV(x.values, yv.values, W, permutations=PERM, seed=SEED)
    q = bh(lb.p_sim)
    BV.append(dict(outcome=o, I_bv=mb.I, p=mb.p_sim, n_HH_p05=int(((lb.q == 1) & (lb.p_sim < .05)).sum()), n_HH_fdr=int(((lb.q == 1) & (q < .05)).sum()),
                   n_LL_p05=int(((lb.q == 3) & (lb.p_sim < .05)).sum())))
    for i in np.where((lb.q == 1) & (lb.p_sim < .05))[0]:
        HH.append(dict(outcome=o, a_code=T.a_code[i], name=T.name[i], p=lb.p_sim[i], q_fdr=q[i], pm=T.pm[i],
                       pm25_hotspot=T.pm25_hotspot[i], source_sector=T.source_sector[i], dist_hwy_km=T.dist_hwy_km[i]))
BV = save(pd.DataFrame(BV), "T_bivariate_moran.csv"); HHt = save(pd.DataFrame(HH), "T_bivariate_lisa_HH.csv")
display(BV.round(3)); display(HHt[HHt.outcome.isin(GROUPS)])
for o in GROUPS:
    r = BV.set_index("outcome").loc[o]; h = HHt[HHt.outcome == o]
    note(f"Bivariate Moran's I PM2.5 × {o}: I = {r.I_bv:.3f}, p = {r.p:.3f}; high–high tambons p<0.05: {int(r.n_HH_p05)} (FDR: {int(r.n_HH_fdr)})"
         + (f" — {', '.join(h.name)}; in PM2.5 hotspot {int(h.pm25_hotspot.sum())}, in source sector {int(h.source_sector.sum())}, ≤2 km of highway {int((h.dist_hwy_km <= 2).sum())}" if len(h) else ""))

## 6 · Step 3 — rate ratios by exposure zone (Poisson, fiscal-year strata, tambon-clustered CI)

In [ ]:
QZ = D.merge(IND, on="a_code"); pmq = QZ.groupby("a_code").pm.transform("mean")
QZ["pm_q4_vs_q1"] = np.where(pmq >= pmq.quantile(.75), 1, np.where(pmq <= pmq.quantile(.25), 0, np.nan))
for v in ["dist_plant_km", "dist_hwy_km"]:
    t1, t3 = QZ[v].quantile(1/3), QZ[v].quantile(2/3)
    QZ[v.replace("dist_", "near_").replace("_km", "")] = np.where(QZ[v] <= t1, 1, np.where(QZ[v] >= t3, 0, np.nan))
QZ["fac_within_5km"] = (QZ.n_fac_5km > 0).astype(int)
CONTRASTS = {"pm_q4_vs_q1": "PM2.5 Q4 vs Q1", "pm25_hotspot": "PM2.5 hotspot vs other", "source_sector": "Source sector vs other",
             "downwind_fac": "Facility-downwind vs other", "near_plant": "Nearest vs farthest tertile, cement plant",
             "fac_within_5km": "≥1 DIW facility within 5 km vs none", "near_hwy": "Nearest vs farthest tertile, highway"}
RR = []
for o in OUTCOMES:
    for v, lab in CONTRASTS.items():
        S = QZ[QZ[v].notna()].copy(); S["e"] = S[v].astype(int); S["yy"] = S[o]
        if S.e.nunique() < 2: continue
        try:
            f = smf.glm("yy ~ e + C(fy)", data=S, family=sm.families.Poisson(), offset=S.log_pop).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(S.a_code)[0]})
            b, se = f.params["e"], f.bse["e"]
            RR.append(dict(outcome=o, contrast=lab, RR=np.exp(b), lo=np.exp(b - 1.96 * se), hi=np.exp(b + 1.96 * se), p=f.pvalues["e"], n_exposed=S[S.e == 1].a_code.nunique(), n_ref=S[S.e == 0].a_code.nunique()))
        except Exception as e: print(o, lab, e)
RR = save(pd.DataFrame(RR), "T_rate_ratios.csv")
display(RR[RR.outcome.isin(GROUPS)].round(2))
for _, r in RR[RR.outcome.isin(["Respiratory", "Cardiovascular"])].iterrows():
    note(f"RR {r.outcome}, {r.contrast}: {r.RR:.2f} ({r.lo:.2f}–{r.hi:.2f}), tambons {r.n_exposed} vs {r.n_ref}")

## 7 · Step 4 — within-tambon IRR by disease, PM2.5 × season, effect modification (NB, tambon fixed effects)

In [ ]:
def nb_fit(df, outcome, extra=""):
    df = df.assign(yy=df[outcome]); f = f"yy ~ pm_w{extra} + C(a_code) + C(month) + C(fy)"
    p0 = smf.glm(f, data=df, family=sm.families.Poisson(), offset=df.log_pop).fit()
    alpha = max(((df.yy - p0.mu) ** 2 - p0.mu).sum() / (p0.mu ** 2).sum(), 1e-3)
    return smf.glm(f, data=df, family=sm.families.NegativeBinomial(alpha=alpha), offset=df.log_pop).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(df.a_code)[0]})
def irr(fit, term="pm_w"):
    b, se = fit.params[term], fit.bse[term]; return np.exp(b), np.exp(b - 1.96 * se), np.exp(b + 1.96 * se), fit.pvalues[term]
DIS = []
for o in OUTCOMES:
    if D[o].sum() == 0: continue
    r = irr(nb_fit(D, o)); DIS.append(dict(outcome=o, visits=int(D[o].sum()), IRR=r[0], lo=r[1], hi=r[2], p=r[3]))
DIS = save(pd.DataFrame(DIS), "T_irr_by_disease.csv"); display(DIS.round(3))
SEA = []
for o in OUTCOMES:
    if D[o].sum() == 0: continue
    S = D.copy()
    for s in ["Winter", "Summer", "Rainy"]: S[f"pm_{s}"] = S.pm_w * (S.season == s)
    fit = nb_fit(S, o, extra="".join(f" + pm_{s}" for s in ["Summer", "Rainy"]))   # winter = reference
    V = fit.cov_params(); b = fit.params
    for s in ["Winter", "Summer", "Rainy"]:
        if s == "Winter": est, var = b["pm_w"], V.loc["pm_w", "pm_w"]
        else: est = b["pm_w"] + b[f"pm_{s}"]; var = V.loc["pm_w", "pm_w"] + V.loc[f"pm_{s}", f"pm_{s}"] + 2 * V.loc["pm_w", f"pm_{s}"]
        SEA.append(dict(outcome=o, season=s, IRR=np.exp(est), lo=np.exp(est - 1.96 * np.sqrt(var)), hi=np.exp(est + 1.96 * np.sqrt(var))))
    w = fit.wald_test(" , ".join(f"pm_{s} = 0" for s in ["Summer", "Rainy"]), scalar=True)
    SEA.append(dict(outcome=o, season="interaction p", IRR=np.nan, lo=np.nan, hi=np.nan, p=float(w.pvalue)))
SEA = save(pd.DataFrame(SEA), "T_irr_season.csv"); display(SEA[SEA.outcome.isin(GROUPS)].round(3))
Q2 = D.merge(IND, on="a_code")
for lam in LAMBDAS_KM: Q2[f"decay_l{lam}_hi"] = (Q2[f"decay_l{lam}"] > Q2.drop_duplicates("a_code")[f"decay_l{lam}"].median()).astype(int)
for r_ in BUFFERS_KM: Q2[f"fac_{r_}km_any"] = (Q2[f"n_fac_{r_}km"] > 0).astype(int)
Q2["near_hwy_2km"] = (Q2.dist_hwy_km <= 2).astype(int)
MODS = {"source_sector": "Source sector (proposal 7.2.4)", "downwind_fac": "Facility-downwind (updated sector)", "pm25_hotspot": "PM2.5 hotspot",
        **{f"fac_{r_}km_any": f"≥1 DIW facility within {r_} km" for r_ in BUFFERS_KM},
        **{f"decay_l{l}_hi": f"Distance-decay index above median, λ = {l} km" for l in LAMBDAS_KM}, "near_hwy_2km": "Highway within 2 km"}
EM = []
for o in ["Respiratory", "Cardiovascular"]:
    for v, lab in MODS.items():
        if Q2[v].nunique() < 2: continue
        S = Q2.copy(); S["pm_x"] = S.pm_w * S[v]
        fit = nb_fit(S, o, extra=" + pm_x"); b, V = fit.params, fit.cov_params()
        e1 = b["pm_w"] + b["pm_x"]; s1 = np.sqrt(V.loc["pm_w", "pm_w"] + V.loc["pm_x", "pm_x"] + 2 * V.loc["pm_w", "pm_x"])
        EM.append(dict(outcome=o, modifier=lab, n_tambon_exposed=int(S.drop_duplicates("a_code")[v].sum()),
                       IRR_exposed=np.exp(e1), lo_exp=np.exp(e1 - 1.96 * s1), hi_exp=np.exp(e1 + 1.96 * s1),
                       IRR_other=np.exp(b["pm_w"]), lo_oth=np.exp(b["pm_w"] - 1.96 * fit.bse["pm_w"]), hi_oth=np.exp(b["pm_w"] + 1.96 * fit.bse["pm_w"]),
                       p_interaction=fit.pvalues["pm_x"]))
EM = save(pd.DataFrame(EM), "T_effect_modification.csv"); display(EM.round(3))
for _, r in DIS.iterrows(): note(f"IRR {r.outcome}: {r.IRR:.2f} ({r.lo:.2f}–{r.hi:.2f})")
for o in GROUPS:
    s = SEA[SEA.outcome == o].set_index("season")
    note(f"PM2.5 × season, {o}: winter {s.loc['Winter','IRR']:.2f}, summer {s.loc['Summer','IRR']:.2f}, rainy {s.loc['Rainy','IRR']:.2f}; interaction p = {s.loc['interaction p','p']:.3f}")
for _, r in EM.iterrows():
    note(f"EM {r.outcome}, {r.modifier} (n={r.n_tambon_exposed}): {r.IRR_exposed:.2f} ({r.lo_exp:.2f}–{r.hi_exp:.2f}) vs {r.IRR_other:.2f} ({r.lo_oth:.2f}–{r.hi_oth:.2f}); p = {r.p_interaction:.3f}")

## 8 · Step 5 — MAE for the out-of-sample prediction (Notebook 20)
The manuscript reports RMSE and Spearman. Apply `pred_metrics` to the observed and predicted tambon rates saved by the leave-one-district-out and leave-last-period-out code in Notebook 20, for the models with and without PM2.5.

In [ ]:
def pred_metrics(obs, pred):
    obs, pred = np.asarray(obs, float), np.asarray(pred, float)
    return dict(RMSE=float(np.sqrt(np.mean((obs - pred) ** 2))), MAE=float(np.mean(np.abs(obs - pred))), Spearman=float(stats.spearmanr(obs, pred).statistic))
# example: pred_metrics(lodo["obs_rate"], lodo["pred_rate_pm"]), pred_metrics(lodo["obs_rate"], lodo["pred_rate_nopm"])

## 9 · Numbers for the manuscript

In [ ]:
open(os.path.join(OUTDIR, "SUMMARY_for_manuscript.txt"), "w", encoding="utf-8").write("\n".join(SUMMARY))
print("\n".join(SUMMARY)); print("\nsaved:", OUTDIR)